# GIA Modeling Tool on Google Colab

Runs the full app (FastAPI backend + built React frontend, one process) inside this
Colab session and opens it in a new browser tab through Colab's port proxy.

**Runtime:** the default CPU runtime is enough. The pipeline's time goes to raster I/O,
contour tracing and the GeoPackage write, none of which a GPU speeds up.

Run the cells top to bottom. Setup (cells 1–4) takes a few minutes the first time in a
session; after that only cells 5–6 are needed if you restart the server.

**Everything in `/content` is deleted when the Colab session ends.** Download your result
zips, and keep large DEMs on Google Drive (cell 4).

In [ ]:
# 1. Settings
REPO_URL = "https://github.com/Levi-Colt/GIA_Modeling_Tool.git"  # @param {type:"string"}
BRANCH = "Master"  # @param {type:"string"}
REPO_DIR = "/content/GIA_Modeling_Tool"  # @param {type:"string"}
PORT = 8765  # @param {type:"integer"}
STORAGE_DIR = "/content/gia_jobs"  # @param {type:"string"}
REBUILD_FRONTEND = False  # @param {type:"boolean"}

import os, subprocess, sys

def run(cmd, cwd=None):
    # Run a command, stream nothing, and fail loudly with its output if it fails.
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")
    return result.stdout

# Get the code (clone once, then fast-forward on later runs)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO_DIR)
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR])
print(run(["git", "log", "--oneline", "-1"], cwd=REPO_DIR))

In [ ]:
# 2. Python dependencies (Colab already has numpy, scipy, pandas, shapely, ...;
#    the requirement files are unpinned, so pip only adds what's missing)
run([sys.executable, "-m", "pip", "install", "-q", "-r",
     os.path.join(REPO_DIR, "setup", "requirements-api.txt")])

import importlib
for mod in ["numpy", "scipy", "rasterio", "geopandas", "shapely", "pyproj",
            "skimage", "psutil", "fastapi", "uvicorn", "multipart"]:
    m = importlib.import_module(mod)
    print(f"{mod:12s} {getattr(m, '__version__', '')}")

In [ ]:
# 3. Build the frontend (needs Node 20+; installs a private copy under /content/node
#    if Colab's Node is missing or too old). Skipped when a build already exists.
import shutil, tarfile, urllib.request

NODE_VERSION = "22.12.0"

def node_major():
    node = shutil.which("node")
    if not node:
        return 0
    return int(run([node, "--version"]).strip().lstrip("v").split(".")[0])

if node_major() < 20:
    node_dir = f"/content/node-v{NODE_VERSION}-linux-x64"
    if not os.path.isdir(node_dir):
        url = f"https://nodejs.org/dist/v{NODE_VERSION}/node-v{NODE_VERSION}-linux-x64.tar.xz"
        archive = "/content/node.tar.xz"
        print(f"Downloading Node {NODE_VERSION} ...")
        urllib.request.urlretrieve(url, archive)
        with tarfile.open(archive) as tar:
            tar.extractall("/content", filter="data")
        os.remove(archive)
    os.environ["PATH"] = os.path.join(node_dir, "bin") + os.pathsep + os.environ["PATH"]
print("node", run([shutil.which("node"), "--version"]).strip())
NPM = shutil.which("npm") or "npm"

frontend = os.path.join(REPO_DIR, "frontend")
dist = os.path.join(frontend, "dist")
if REBUILD_FRONTEND or not os.path.isfile(os.path.join(dist, "index.html")):
    print("npm ci ... (a minute or two)")
    run([NPM, "ci", "--no-audit", "--no-fund"], cwd=frontend)
    run([NPM, "run", "build"], cwd=frontend)
print("frontend build:", "OK" if os.path.isfile(os.path.join(dist, "index.html")) else "MISSING")

### 4. Your DEM

Two ways to give the app a DEM, both supported by the app's **DEM** section:

* **Drag and drop** a GeoTIFF onto the app. Fine for small files; large uploads go
  through Colab's proxy and may be slow or rejected.
* **Type a server-side path** (recommended for anything large). The app runs in this
  Colab VM, so it reads files here directly, for example a file you dropped into the
  file browser on the left (`/content/my_dem.tif`) or one on Google Drive after running
  the cell below (`/content/drive/MyDrive/dems/my_dem.tif`).

In [ ]:
# 4. (Optional) Mount Google Drive so the app can read DEMs from it by path
MOUNT_DRIVE = False  # @param {type:"boolean"}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# 5. Start (or restart) the app server in the background
import time, urllib.request

LOG_PATH = "/content/gia_server.log"

def stop_server():
    proc = globals().get("server_proc")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()

def server_log(lines=40):
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            print("".join(f.readlines()[-lines:]))

stop_server()
os.makedirs(STORAGE_DIR, exist_ok=True)
env = dict(os.environ, GIA_STORAGE_DIR=STORAGE_DIR)
log_file = open(LOG_PATH, "w")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", str(PORT)],
    cwd=REPO_DIR, env=env, stdout=log_file, stderr=subprocess.STDOUT,
)

health_url = f"http://127.0.0.1:{PORT}/api/health"
for _ in range(120):
    if server_proc.poll() is not None:
        server_log()
        raise RuntimeError("The server exited during startup; its log is above.")
    try:
        with urllib.request.urlopen(health_url, timeout=2) as r:
            if r.status == 200:
                print(f"Server is up on port {PORT} (pid {server_proc.pid}).")
                break
    except OSError:
        time.sleep(1)
else:
    server_log()
    raise RuntimeError("The server didn't answer /api/health within 2 minutes; its log is above.")

In [ ]:
# 6. Open the app (opens in a new tab; it only works in this browser, signed in to this Colab)
try:
    from google.colab import output
    from google.colab.output import eval_js
    print("App URL:", eval_js(f"google.colab.kernel.proxyPort({PORT})"))
    output.serve_kernel_port_as_window(PORT, path="/")
except ImportError:
    print(f"Not running in Colab; open http://127.0.0.1:{PORT}/ instead.")

In [ ]:
# 7. Server log (run whenever something goes wrong)
server_log(80)

In [ ]:
# 8. Stop the server
stop_server()
print("Stopped.")

### Troubleshooting

| Symptom | Fix |
|---|---|
| The app tab shows a Google error or keeps loading | Open it from cell 6 in the same browser profile that's signed in to Colab; some browsers block the proxy's cookie in private windows. |
| `/` shows `{"detail":"Not Found"}` | The frontend build is missing: tick `REBUILD_FRONTEND` in cell 1, rerun cells 1, 3 and 5. |
| A large drag-and-drop upload fails | Use a typed path instead (cell 4). |
| A run dies or the session restarts | Out of memory: Colab's free tier has about 12 GB of RAM. Untick *Include tilted DEM raster in output* in the Output section, or use a smaller DEM. |
| Anything else | Run cell 7 and read the end of the log. |